# Stock Price Prediction using LSTM neural networks

In [23]:
import pandas as pd
import yfinance as yf
import datetime
from datetime import timedelta,date
from sklearn.model_selection import train_test_split

In [24]:
today = date.today()
d1 = today.strftime("%Y-%m-%d")
end_date = d1
d2 = date.today() - timedelta(days=5000)
start_date = d2

In [25]:
data = yf.download("AAPL",start=start_date,end=end_date,progress=False)

/tmp/ipython-input-2304449577.py:1: FutureWarning:

YF.download() has changed argument auto_adjust default to True



In [26]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2011-12-22,11.964515,11.981927,11.890966,11.917983,202358800
2011-12-23,12.108015,12.115820,11.992738,11.998742,269399200
2011-12-27,12.204082,12.280933,12.098710,12.101113,265076000
2011-12-28,12.087300,12.255713,12.048274,12.214886,228662000
2011-12-29,12.161750,12.177661,12.023358,12.110116,215978000


In [27]:
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data = data.reset_index()

data['Date'] = data.index

data = data[['Date','Open','High','Low','Close','Volume']]
# data.reset_index(drop=True,inplace=True)

In [28]:
data.head()

,Date,Open,High,Low,Close,Volume
0,0,11.917983,11.981927,11.890966,11.964515,202358800
1,1,11.998742,12.115820,11.992738,12.108015,269399200
2,2,12.101113,12.280933,12.098710,12.204082,265076000
3,3,12.214886,12.255713,12.048274,12.087300,228662000
4,4,12.110116,12.177661,12.023358,12.161750,215978000


In [29]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data['Date'],
                                        open=data['Open'],
                                        high=data['High'],
                                        low=data['Low'],
                                        close=data['Close'])])
figure.update_layout(title='Apple Stock Price analysis',
                     xaxis_rangeslider_visible=False)
figure.show()

In [30]:
correlation = data.corr()
correlation['Close'].sort_values(ascending=False)

,Close
Close,1.000000
High,0.999883
Low,0.999878
Open,0.999745
Date,0.926993
Volume,-0.528665


In [31]:
x = data[['Open','High','Low','Volume']]
y = data['Close']
x = x.to_numpy()
y = y.to_numpy()
y = y.reshape(-1,1)

In [33]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [34]:
from keras.models import Sequential
from keras.layers import Dense,LSTM

In [36]:
model = Sequential()
model.add(LSTM(128,return_sequences=True,input_shape=(X_train.shape[1],1)))
model.add(LSTM(128,return_sequences=True))
model.add(Dense(25))
model.add(Dense(1))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 128)         │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 4, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4, 25)          │         3,225 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4, 1)           │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,395 (786.70 KB)

 Trainable params: 201,395 (786.70 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
model.compile(optimizer='adam',loss="mean_squared_error")
model.fit(X_train,y_train,batch_size=1,epochs=30)

Epoch 1/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 1537.6891
Epoch 2/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 31s 11ms/step - loss: 40.6035
Epoch 3/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 33s 8ms/step - loss: 21.8368
Epoch 4/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - loss: 51.2952
Epoch 5/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 36s 8ms/step - loss: 22.3583
Epoch 6/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - loss: 21.7232
Epoch 7/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 23s 8ms/step - loss: 24.2438
Epoch 8/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 23.1222
Epoch 9/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 42s 9ms/step - loss: 25.8258
Epoch 10/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 18.2268
Epoch 11/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 42s 8ms/step - loss: 14.2795
Epoch 12/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 19.9096
Epoch 13/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - loss: 16.6108
Epoch 14/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 16.18

In [39]:
import numpy as np
features = np.array([[177.089996,180.419998,177.070007,74919600]])
model.predict(features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


array([[[174.21594],
        [179.58026],
        [177.87627],
        [177.75378]]], dtype=float32)